# MAI-142 deterministic reference model export

This notebook documents the reproducible Python/Colab path for the local ONNX reference artifact. It uses deterministic synthetic data only. The result is a reference-runtime contract fixture, not a statistically useful, production-certified, or regulatory model.

The six input features are ordered as `ebitda`, `revenue`, `totalAssets`, `totalLiabilities`, `operatingCashFlow`, and `historicalVolatility`. The model emits one `predictions` tensor with four ordered values: `insolvencyRisk`, `churnProbability`, `dcfVolatility`, and `industry4Score`.

In [ ]:
# Run from the repository root in a clean Python 3.10+ or Colab runtime.
!python -m pip install --quiet numpy==1.26.4 onnx==1.16.1 onnxruntime==1.18.1

In [ ]:
# The canonical exporter owns the fixed seeds, full-batch training, ONNX opset,
# dynamic [batch, 6] -> [batch, 4] contract, q8 export, and manifest/checksum.
from pathlib import Path

EXPORTER = Path('scripts/export_reference_model.py')
assert EXPORTER.is_file(), f'Missing exporter: {EXPORTER}'
print(EXPORTER.resolve())

In [ ]:
# Export twice in clean temporary directories and compare SHA-256 checksums.
!python scripts/export_reference_model.py --check-reproducibility

In [ ]:
# Produce the committed reference artifact, manifest, and checksum sidecar.
!python scripts/export_reference_model.py --output-dir models

In [ ]:
import hashlib
import json
from pathlib import Path

artifact = Path('models/ma_classifier_q8.onnx')
manifest = json.loads(Path('models/ma_classifier_q8.manifest.json').read_text())
checksum = hashlib.sha256(artifact.read_bytes()).hexdigest()
assert checksum == manifest['artifact']['sha256']
assert manifest['input']['shape'] == ['batch_size', 6]
assert manifest['output']['shape'] == ['batch_size', 4]
assert manifest['artifact']['readinessClassification'] == 'reference-only'
print(json.dumps({'sha256': checksum, 'manifest': manifest}, indent=2))

## Limitations and rollback

The dataset is synthetic and generated from seeded normal samples plus deterministic rule-derived targets. It is not production-qualified and has no evidence of statistical usefulness or regulatory suitability. The local rollback unit is the pair `models/ma_classifier_q8.onnx` and `models/ma_classifier_q8.manifest.json` (with its `.sha256` sidecar); revert that pair together.